# Experiment 006 — Real TinyStories GPU Development Pilot

This notebook starts real on-policy hard-path language-model training, not just a CPU fixture.
It is explicitly **UNREGISTERED DEVELOPMENT ONLY**: no confirmation data, no scientific approval,
no GitHub result publication. The original formal Notebook and Gate 1 registration remain unchanged.

**Kaggle:** Internet On; accelerator **GPU T4 x2** for train_pair. Leave MODE=train_pair,
ARMS=(R0,R1), TRAIN_TOKENS=1_000_000 for the first real development run.
Switch ARMS=(R2,M0) for the second pair. For a full 10M-token pilot,
change TRAIN_TOKENS to 10_000_000; a separate directory preserves the distinct schedule.

All weights, optimizer checkpoints, data/Tokenizer hashes, learning curves and decode profiles
are stored under /kaggle/working. Save Kaggle outputs to retain checkpoints between sessions.
Seed 6060611 is consumed for pilot development and must never become a confirmation seed.


In [ ]:
from pathlib import Path
import os, sys, subprocess, threading

# REAL TinyStories development-only pilot. No formal protocol/confirmation.
MODE = "train_pair"   # preflight | prepare | train_pair | train_one | profile
ARMS = ("R0", "R1")  # second comparison: ("R2", "M0")
SINGLE_ARM = "R2"
TRAIN_TOKENS = 1_000_000  # 1_000_000 or 10_000_000; separate workdir when changing
DEV_SEED = 6060611  # pilot-only, not a formal confirmation seed
SOURCE_COMMIT = "629edbd72170639e0773811ea5f25256554edc62"
ROOT = Path("/kaggle/working/minicells-exp006-pilot-code-dedup-v1")
WORK = Path(f"/kaggle/working/minicells-exp006-pilot-{TRAIN_TOKENS}-dedup-v1")
REMOTE = "https://github.com/ArcheLabs/mini-cells.git"

if MODE not in {"preflight", "prepare", "train_pair", "train_one", "profile"}:
    raise ValueError("unknown development pilot mode")
if MODE in {"train_pair", "train_one", "profile"}:
    import torch
    required = 2 if MODE == "train_pair" else 1
    if torch.cuda.device_count() < required:
        raise RuntimeError(f"Select GPU T4 x2 (requires {required} GPUs)")
if MODE == "train_pair" and (len(ARMS) != 2 or len(set(ARMS)) != 2):
    raise ValueError("Choose two distinct experimental arms")

# These libraries process only the pinned train-split source shard.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "datasets>=3,<5", "tokenizers>=0.20,<0.23",
                "huggingface-hub>=0.25,<2"], check=True)

if not ROOT.exists():
    ROOT.mkdir(parents=True)
    subprocess.run(["git", "init"], cwd=ROOT, check=True, stdout=subprocess.DEVNULL)
    subprocess.run(["git", "remote", "add", "origin", REMOTE], cwd=ROOT, check=True)
    subprocess.run(["git", "fetch", "--depth=1", "origin", SOURCE_COMMIT], cwd=ROOT, check=True)
    subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=ROOT, check=True)
else:
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
    if actual != SOURCE_COMMIT:
        raise RuntimeError("Existing checkout has another source SHA. Use a fresh ROOT; do not overwrite it.")
env_base = dict(os.environ)
impl = ROOT / "research/08-hybrid-clm/experiments/006-path-first-clm/implementation"
env_base["PYTHONPATH"] = str(impl) + os.pathsep + str(ROOT)
env_base["PYTHONUNBUFFERED"] = "1"
env_base["TOKENIZERS_PARALLELISM"] = "false"
print({"mode": MODE, "source_commit": SOURCE_COMMIT, "work": str(WORK),
       "scientific_decision": False, "formal_confirmation": False})


In [ ]:
def cmd(mode, arm=None):
    if mode == "preflight":
        return [sys.executable, str(ROOT/"scripts/research/run_experiment_006.py"),
                "--mode", "preflight"]
    call = [sys.executable, "-m", "exp006.development_pilot",
            "--mode", mode, "--workdir", str(WORK),
            "--tokens", str(TRAIN_TOKENS)]
    if arm:
        call += ["--arm", arm, "--seed", str(DEV_SEED),
                 "--source-commit", SOURCE_COMMIT, "--device", "cuda:0"]
    return call

def stream(command, gpu, label):
    environment = dict(env_base, CUDA_VISIBLE_DEVICES=str(gpu))
    proc = subprocess.Popen(command, cwd=ROOT, env=environment,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    def reader():
        for line in proc.stdout:
            print("[" + label + "] " + line.rstrip(), flush=True)
    thread = threading.Thread(target=reader, daemon=True)
    thread.start()
    return proc, thread

if MODE == "preflight":
    subprocess.run(cmd("preflight"), cwd=ROOT, env=env_base, check=True)
else:
    # Download train shard and construct verified, disjoint train/development assets.
    subprocess.run(cmd("prepare"), cwd=ROOT, env=env_base, check=True)
    selected = ARMS if MODE in ("train_pair", "profile") else (SINGLE_ARM,)
    if MODE != "prepare":
        if MODE in ("train_pair", "train_one"):
            workers = [(arm, *stream(cmd("train", arm), i, arm))
                       for i, arm in enumerate(selected)]
            failed = []
            for arm, proc, thread in workers:
                result = proc.wait()
                thread.join()
                if result:
                    failed.append((arm, result))
            if failed:
                raise RuntimeError(f"Training failed {failed}; retain checkpoints and rerun unchanged")
        for arm in selected:
            proc, thread = stream(cmd("profile", arm), 0, arm + "-decode")
            result = proc.wait()
            thread.join()
            if result:
                raise RuntimeError(f"Decode profiling failed for {arm}; training checkpoints are retained")
        print({"status": "DEV_PILOT_COMPLETE", "arms": selected,
               "results": str(WORK/"runs"/("seed-"+str(DEV_SEED))),
               "scientific_decision": False})
